In [1]:
import pandas as pd

print("Versión de pandas:", pd.__version__)
print("Entorno de Python funcionando correctamente")

Versión de pandas: 3.1.0
Entorno de Python funcionando correctamente


In [2]:
# Cargamos los datos de MiBici correspondientes a enero de 2024
# Especificamos latin1 para leerlo como UTF-8

df = pd.read_csv(
    "2024/datos_abiertos_2024_01.csv",
    encoding="latin1"
)

# Mostramos los primeros cinco registros para verificar contenido, o columnas y valores
df.head()

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,31789291,640715,M,1984.0,2024-01-01 00:05:25,2024-01-01 00:29:37,294,271
1,31789292,2353010,F,1995.0,2024-01-01 00:07:23,2024-01-01 00:16:37,182,254
2,31789293,1556365,M,1984.0,2024-01-01 00:07:31,2024-01-01 00:16:37,182,254
3,31789294,2626233,M,1994.0,2024-01-01 00:07:56,2024-01-01 00:31:51,35,154
4,31789295,2602006,M,1998.0,2024-01-01 00:08:09,2024-01-01 00:31:51,35,154


In [3]:
# Examinamos la estructura y los tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 391591 entries, 0 to 391590
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Viaje_Id           391591 non-null  int64  
 1   Usuario_Id         391591 non-null  int64  
 2   Genero             389556 non-null  str    
 3   Año_de_nacimiento  391202 non-null  float64
 4   Inicio_del_viaje   391591 non-null  str    
 5   Fin_del_viaje      391591 non-null  str    
 6   Origen_Id          391591 non-null  int64  
 7   Destino_Id         391591 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 23.9 MB


In [5]:
# Calculamos los valores faltantes y su porcentaje por variable
valores_faltantes = df.isna().sum()
porcentaje_faltantes = df.isna().mean() * 100

# Presentamos los resultados en una tabla
resumen_faltantes = pd.DataFrame({
    "Valores faltantes": valores_faltantes,
    "Porcentaje (%)": porcentaje_faltantes.round(3)
})

resumen_faltantes

,Valores faltantes,Porcentaje (%)
Viaje_Id,0,0.000
Usuario_Id,0,0.000
Genero,2035,0.520
Año_de_nacimiento,389,0.099
Inicio_del_viaje,0,0.000
Fin_del_viaje,0,0.000
Origen_Id,0,0.000
Destino_Id,0,0.000


### Justificación metodológica de la validación

Evaluamos el efecto de eliminar registros sin información sobre el año de nacimiento, conforme a una de las transformaciones presentes en el módulo de limpieza del proyecto MiBici.

Nuestro objetivo es determinar si esta operación implica una pérdida de observaciones relevantes para la predicción de demanda horaria por estación. Para ello, comparamos los registros originales con los conservados después del filtro y examinamos su distribución por estación de origen.

Adaptamos procedimientos de análisis exploratorio y limpieza de datos estudiados en los notebooks 1.3 y 1.4 del curso TC5044. El análisis específico del impacto sobre la demanda constituye una aplicación al problema del proyecto.

**Fuentes**: datos abiertos de MiBici (2024–2025); notebooks de ejemplo (1.3_WineData_EDA_Example.ipynb y 1.4_WineQuality_DataCleaning.ipynb) del curso; módulo `mibici/dataset.py` del repositorio del equipo.

**Asistencia de IA:** OpenAI GPT-6, utilizado como apoyo para adaptar procedimientos de pandas, explicar instrucciones y estructurar la validación. Los resultados se obtienen mediante ejecución local sobre los CSV originales.

In [6]:
# Comparamos el número de viajes antes y después de eliminar
# los registros sin año de nacimiento

viajes_originales = len(df)

df_sin_nulos_nacimiento = df.dropna(
    subset=["Año_de_nacimiento"]
)

viajes_conservados = len(df_sin_nulos_nacimiento)
viajes_eliminados = viajes_originales - viajes_conservados

# Presentamos los resultados de la comparación
print("Viajes originales:", viajes_originales)
print("Viajes conservados:", viajes_conservados)
print("Viajes eliminados:", viajes_eliminados)
print(
    "Porcentaje eliminado:",
    round(viajes_eliminados / viajes_originales * 100, 3),
    "%"
)

Viajes originales: 391591
Viajes conservados: 391202
Viajes eliminados: 389
Porcentaje eliminado: 0.099 %


In [7]:
# Identificamos los viajes que se eliminarían por no tener
# registrado el año de nacimiento

viajes_sin_nacimiento = df[
    df["Año_de_nacimiento"].isna()
]

# Contamos los viajes afectados por estación de origen
estaciones_afectadas = (
    viajes_sin_nacimiento["Origen_Id"]
    .value_counts()
)

# Mostramos las diez estaciones con más viajes afectados
estaciones_afectadas.head(10)

Origen_Id
179    32
69     29
157    24
253    23
315    22
283    20
281    19
4      19
267    15
249    14
Name: count, dtype: int64

In [8]:
# Calculamos el total de viajes por estación de origen
viajes_por_estacion = df["Origen_Id"].value_counts()

# Comparamos los viajes totales con los que se eliminarían
impacto_estaciones = pd.DataFrame({
    "Viajes totales": viajes_por_estacion,
    "Viajes eliminados": estaciones_afectadas
})

# Sustituimos por cero las estaciones sin viajes eliminados
impacto_estaciones["Viajes eliminados"] = (
    impacto_estaciones["Viajes eliminados"]
    .fillna(0)
    .astype(int)
)

# Calculamos el porcentaje de pérdida por estación
impacto_estaciones["Porcentaje eliminado"] = (
    impacto_estaciones["Viajes eliminados"]
    / impacto_estaciones["Viajes totales"] * 100
).round(3)

# Identificamos las diez estaciones con mayor pérdida proporcional
impacto_estaciones.sort_values(
    "Porcentaje eliminado",
    ascending=False
).head(10)

,Viajes totales,Viajes eliminados,Porcentaje eliminado
Origen_Id,,,
315,428,22,5.140
384,415,12,2.892
325,378,8,2.116
179,1915,32,1.671
253,1853,23,1.241
283,1738,20,1.151
267,1377,15,1.089
249,1368,14,1.023
69,2881,29,1.007


### Resultados de la validación del año de nacimiento

Analizamos 391,591 viajes de MiBici correspondientes a enero de 2024. Identificamos 389 registros (0.099 %) sin información sobre el año de nacimiento.

Al evaluar el efecto de eliminarlos, observamos diferencias entre estaciones. Por ejemplo, la estación 315 perdería 22 de sus 428 viajes (5.140 %), mientras que la estación 384 perdería 12 de 415 (2.892 %).

Concluimos que la eliminación de registros sin año de nacimiento podría afectar los conteos de demanda de algunas estaciones. Por ello, recomendamos conservar estos viajes cuando dispongan de fechas y estaciones válidas, dado que el año de nacimiento no es indispensable para construir nuestra variable objetivo.

Estos resultados corresponden únicamente a enero de 2024 y deberán contrastarse con otros periodos antes de generalizar la conclusión.

In [9]:
# Convertimos las fechas originales a valores de fecha y hora
inicio = pd.to_datetime(
    df["Inicio_del_viaje"],
    errors="coerce",
    format="%Y-%m-%d %H:%M:%S"
)

fin = pd.to_datetime(
    df["Fin_del_viaje"],
    errors="coerce",
    format="%Y-%m-%d %H:%M:%S"
)

# Identificamos fechas inválidas y viajes con duración negativa
fechas_inicio_invalidas = inicio.isna().sum()
fechas_fin_invalidas = fin.isna().sum()

duraciones_negativas = (
    (fin < inicio) & inicio.notna() & fin.notna()
).sum()

# Presentamos los resultados
print("Fechas de inicio inválidas:", fechas_inicio_invalidas)
print("Fechas de fin inválidas:", fechas_fin_invalidas)
print("Viajes con duración negativa:", duraciones_negativas)

Fechas de inicio inválidas: 0
Fechas de fin inválidas: 0
Viajes con duración negativa: 0


In [10]:
# Verificamos si existen identificadores de viaje duplicados
viajes_id_duplicados = df["Viaje_Id"].duplicated().sum()

# Verificamos si existen filas completamente idénticas
filas_duplicadas = df.duplicated().sum()

# Presentamos los resultados
print("Identificadores de viaje duplicados:", viajes_id_duplicados)
print("Filas completamente duplicadas:", filas_duplicadas)

Identificadores de viaje duplicados: 0
Filas completamente duplicadas: 0


In [11]:
from pathlib import Path

# Localizamos los archivos CSV de 2024 y 2025
archivos_2024 = sorted(Path("2024").glob("datos_abiertos_*.csv"))
archivos_2025 = sorted(Path("2025").glob("datos_abiertos_*.csv"))

# Contamos los archivos disponibles por año
print("Archivos de 2024:", len(archivos_2024))
print("Archivos de 2025:", len(archivos_2025))
print("Total de archivos:", len(archivos_2024) + len(archivos_2025))

Archivos de 2024: 12
Archivos de 2025: 12
Total de archivos: 24


In [12]:
# Reunimos los archivos de ambos años
archivos = archivos_2024 + archivos_2025

# Preparamos una lista para almacenar los resultados mensuales
resultados = []

# Examinamos cada archivo por separado
for archivo in archivos:
    datos = pd.read_csv(archivo, encoding="latin1")

    # Identificamos la columna de año de nacimiento
    columnas_nacimiento = [
        col for col in datos.columns
        if "nacimiento" in col.lower()
    ]

    # Calculamos los indicadores básicos de calidad
    resultados.append({
        "Archivo": archivo.name,
        "Viajes": len(datos),
        "Género faltante": datos["Genero"].isna().sum(),
        "Nacimiento faltante": (
            datos[columnas_nacimiento[0]].isna().sum()
            if columnas_nacimiento else None
        ),
        "Viaje_Id duplicados": datos["Viaje_Id"].duplicated().sum()
    })

# Construimos una tabla con los resultados
resumen_mensual = pd.DataFrame(resultados)

# Mostramos los resultados de los 24 meses
resumen_mensual

,Archivo,Viajes,Género faltante,Nacimiento faltante,Viaje_Id duplicados
0,datos_abiertos_2024_01.csv,391591,2035,389,0
1,datos_abiertos_2024_02.csv,383732,2231,333,0
2,datos_abiertos_2024_03.csv,390513,2097,283,0
3,datos_abiertos_2024_04.csv,405308,2123,271,0
4,datos_abiertos_2024_05.csv,410806,2494,254,0
5,datos_abiertos_2024_06.csv,365000,2191,134,0
6,datos_abiertos_2024_07.csv,364177,2098,215,0
7,datos_abiertos_2024_08.csv,394894,2395,167,0
8,datos_abiertos_2024_09.csv,390072,2750,129,0
9,datos_abiertos_2024_10.csv,418690,2910,169,0


In [13]:
# Identificamos el año de cada archivo
resumen_mensual["Año"] = (
    resumen_mensual["Archivo"]
    .str.extract(r"datos_abiertos_(\d{4})_")[0]
    .astype(int)
)

# Agrupamos los indicadores de calidad por año
resumen_anual = (
    resumen_mensual
    .groupby("Año")[
        [
            "Viajes",
            "Género faltante",
            "Nacimiento faltante",
            "Viaje_Id duplicados"
        ]
    ]
    .sum()
)

# Calculamos los porcentajes de valores faltantes
resumen_anual["% género faltante"] = (
    resumen_anual["Género faltante"]
    / resumen_anual["Viajes"] * 100
).round(3)

resumen_anual["% nacimiento faltante"] = (
    resumen_anual["Nacimiento faltante"]
    / resumen_anual["Viajes"] * 100
).round(3)

# Presentamos el resumen de ambos años
resumen_anual

,Viajes,Género faltante,Nacimiento faltante,Viaje_Id duplicados,% género faltante,% nacimiento faltante
Año,,,,,,
2024,4645921,28946,2667,0,0.623,0.057
2025,4532032,40789,1105,0,0.900,0.024


In [14]:
# Revisamos los nombres de las columnas en los 24 archivos
estructura_archivos = []

for archivo in archivos:
    # Leemos únicamente los encabezados, sin cargar los viajes
    columnas = pd.read_csv(
        archivo,
        encoding="latin1",
        nrows=0
    ).columns.tolist()

    # Registramos las columnas de cada archivo
    estructura_archivos.append({
        "Archivo": archivo.name,
        "Número de columnas": len(columnas),
        "Columnas": columnas
    })

# Identificamos las estructuras diferentes
estructuras = {}

for registro in estructura_archivos:
    estructura = tuple(registro["Columnas"])
    estructuras.setdefault(estructura, []).append(registro["Archivo"])

# Presentamos cuántas estructuras encontramos
print("Estructuras de columnas diferentes:", len(estructuras))

for numero, (columnas, archivos_asociados) in enumerate(
    estructuras.items(), start=1
):
    print(f"\nEstructura {numero}:")
    print("Archivos:", len(archivos_asociados))
    print("Columnas:", columnas)
    print("Primer archivo:", archivos_asociados[0])

Estructuras de columnas diferentes: 3

Estructura 1:
Archivos: 12
Columnas: ('Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id')
Primer archivo: datos_abiertos_2024_01.csv

Estructura 2:
Archivos: 2
Columnas: ('Viaje_Id', 'Usuario_Id', 'Genero', 'A}äe_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id')
Primer archivo: datos_abiertos_2024_02.csv

Estructura 3:
Archivos: 10
Columnas: ('Viaje_Id', 'Usuario_Id', 'Genero', 'AÃ±o_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id')
Primer archivo: datos_abiertos_2024_07.csv


In [15]:
# Identificamos los archivos asociados a cada estructura
for numero, (columnas, archivos_asociados) in enumerate(
    estructuras.items(), start=1
):
    print(f"\nEstructura {numero}:")
    print("Archivos asociados:")

    for nombre_archivo in archivos_asociados:
        print(" -", nombre_archivo)


Estructura 1:
Archivos asociados:
 - datos_abiertos_2024_01.csv
 - datos_abiertos_2024_03.csv
 - datos_abiertos_2024_04.csv
 - datos_abiertos_2024_06.csv
 - datos_abiertos_2024_08.csv
 - datos_abiertos_2024_09.csv
 - datos_abiertos_2024_10.csv
 - datos_abiertos_2024_11.csv
 - datos_abiertos_2024_12.csv
 - datos_abiertos_2025_01.csv
 - datos_abiertos_2025_02.csv
 - datos_abiertos_2025_07.csv

Estructura 2:
Archivos asociados:
 - datos_abiertos_2024_02.csv
 - datos_abiertos_2024_05.csv

Estructura 3:
Archivos asociados:
 - datos_abiertos_2024_07.csv
 - datos_abiertos_2025_03.csv
 - datos_abiertos_2025_04.csv
 - datos_abiertos_2025_05.csv
 - datos_abiertos_2025_06.csv
 - datos_abiertos_2025_08.csv
 - datos_abiertos_2025_09.csv
 - datos_abiertos_2025_10.csv
 - datos_abiertos_2025_11.csv
 - datos_abiertos_2025_12.csv


In [16]:
# Verificamos la normalización del encabezado de nacimiento
resultados_normalizacion = []

for archivo in archivos:
    # Leemos únicamente los encabezados originales
    columnas = pd.read_csv(
        archivo,
        encoding="latin1",
        nrows=0
    ).columns.tolist()

    # Identificamos la columna correspondiente al año de nacimiento
    columnas_nacimiento = [
        col for col in columnas
        if "nacimiento" in col.lower()
    ]

    # Comprobamos que exista exactamente una columna de nacimiento
    if len(columnas_nacimiento) != 1:
        raise ValueError(
            f"Encabezado ambiguo en {archivo.name}: "
            f"{columnas_nacimiento}"
        )

    # Sustituimos únicamente el encabezado identificado
    columnas_normalizadas = [
        "Año_de_nacimiento" if col == columnas_nacimiento[0] else col
        for col in columnas
    ]

    # Registramos el resultado de la comprobación
    resultados_normalizacion.append({
        "Archivo": archivo.name,
        "Columnas": len(columnas_normalizadas),
        "Nacimiento normalizado": (
            columnas_normalizadas.count("Año_de_nacimiento") == 1
        ),
        "Estructura válida": columnas_normalizadas == [
            "Viaje_Id",
            "Usuario_Id",
            "Genero",
            "Año_de_nacimiento",
            "Inicio_del_viaje",
            "Fin_del_viaje",
            "Origen_Id",
            "Destino_Id"
        ]
    })

# Presentamos el resultado consolidado de la validación
validacion_normalizacion = pd.DataFrame(resultados_normalizacion)

print("Archivos revisados:", len(validacion_normalizacion))
print(
    "Archivos con estructura válida:",
    validacion_normalizacion["Estructura válida"].sum()
)
print(
    "Archivos con estructura inválida:",
    (~validacion_normalizacion["Estructura válida"]).sum()
)

Archivos revisados: 24
Archivos con estructura válida: 24
Archivos con estructura inválida: 0


### Validación de formatos de fecha

Durante nuestra revisión identificamos dos representaciones de fecha en los archivos de MiBici: año-mes-día y día/mes/año.

Comprobamos que la conversión mediante `format="mixed"` puede interpretar incorrectamente las fechas ambiguas cuando no se especifica el orden del día y el mes.

Por ejemplo, `12/09/2024 23:39` fue interpretada como el 9 de diciembre de 2024, aunque corresponde al 12 de septiembre de 2024.

Por ello, decidimos utilizar formatos explícitos según el archivo de origen. Esta decisión permite evitar interpretaciones ambiguas y preservar la integridad temporal de los registros utilizados para calcular la demanda horaria.

La prueba siguiente documenta la diferencia entre ambas interpretaciones.

In [17]:
# Comparamos dos interpretaciones de una fecha original
fecha_original = "12/09/2024 23:39"

# Interpretamos la fecha con la configuración predeterminada
fecha_automatica = pd.to_datetime(
    fecha_original,
    format="mixed"
)

# Interpretamos la fecha indicando día, mes y año
fecha_dia_primero = pd.to_datetime(
    fecha_original,
    format="%d/%m/%Y %H:%M"
)

# Comparamos los resultados
print("Fecha original:", fecha_original)
print("Interpretación automática:", fecha_automatica)
print("Interpretación día/mes/año:", fecha_dia_primero)

Fecha original: 12/09/2024 23:39
Interpretación automática: 2024-12-09 23:39:00
Interpretación día/mes/año: 2024-09-12 23:39:00


In [18]:
# Revisamos ejemplos de fechas en los 24 archivos
# sin cargar todos los registros en memoria

for archivo in archivos:
    muestra = pd.read_csv(
        archivo,
        encoding="latin1",
        usecols=["Inicio_del_viaje"],
        nrows=3
    )

    print(
        archivo.name,
        "→",
        muestra["Inicio_del_viaje"].tolist()
    )
    

datos_abiertos_2024_01.csv → ['2024-01-01 00:05:25', '2024-01-01 00:07:23', '2024-01-01 00:07:31']
datos_abiertos_2024_02.csv → ['2024-02-01 00:01:46', '2024-02-01 00:02:26', '2024-02-01 00:02:36']
datos_abiertos_2024_03.csv → ['2024-03-01 00:00:22', '2024-03-01 00:00:52', '2024-03-01 00:01:26']
datos_abiertos_2024_04.csv → ['2024-04-01 00:00:48', '2024-04-01 00:02:10', '2024-04-01 00:02:18']
datos_abiertos_2024_05.csv → ['2024-05-01 00:01:35', '2024-05-01 00:02:00', '2024-05-01 00:02:29']
datos_abiertos_2024_06.csv → ['2024-06-01 00:00:17', '2024-06-01 00:00:26', '2024-06-01 00:01:02']
datos_abiertos_2024_07.csv → ['2024-07-01 00:00:34', '2024-07-01 00:01:09', '2024-07-01 00:03:32']
datos_abiertos_2024_08.csv → ['2024-08-01 00:00:09', '2024-08-01 00:01:08', '2024-08-01 00:01:16']
datos_abiertos_2024_09.csv → ['01/09/2024 0:00', '01/09/2024 0:02', '01/09/2024 0:02']
datos_abiertos_2024_10.csv → ['01/10/2024 0:00', '01/10/2024 0:01', '01/10/2024 0:01']
datos_abiertos_2024_11.csv → ['202

In [19]:
# Seleccionamos los dos archivos con formato día/mes/año
archivos_especiales = [
    "2024/datos_abiertos_2024_09.csv",
    "2024/datos_abiertos_2024_10.csv"
]

# Preparamos los resultados de la validación
resultados_formato = []

for ruta in archivos_especiales:

    # Leemos únicamente las fechas necesarias
    datos = pd.read_csv(
        ruta,
        encoding="latin1",
        usecols=["Inicio_del_viaje", "Fin_del_viaje"]
    )

    # Convertimos las fechas usando un formato explícito
    inicio = pd.to_datetime(
        datos["Inicio_del_viaje"],
        format="%d/%m/%Y %H:%M",
        errors="coerce"
    )

    fin = pd.to_datetime(
        datos["Fin_del_viaje"],
        format="%d/%m/%Y %H:%M",
        errors="coerce"
    )

    # Comprobamos fechas inválidas y duraciones negativas
    resultados_formato.append({
        "Archivo": ruta,
        "Inicio inválido": inicio.isna().sum(),
        "Fin inválido": fin.isna().sum(),
        "Duración negativa": (
            (fin < inicio) & inicio.notna() & fin.notna()
        ).sum()
    })

# Presentamos los resultados
pd.DataFrame(resultados_formato)

,Archivo,Inicio inválido,Fin inválido,Duración negativa
0,2024/datos_abiertos_2024_09.csv,0,0,0
1,2024/datos_abiertos_2024_10.csv,0,0,0


In [20]:
# Preparamos una lista para registrar los resultados mensuales
resultados_temporales = []

for archivo in archivos:

    # Seleccionamos el formato de fecha según el archivo
    if archivo.name in [
        "datos_abiertos_2024_09.csv",
        "datos_abiertos_2024_10.csv"
    ]:
        formato_fecha = "%d/%m/%Y %H:%M"
    else:
        formato_fecha = "%Y-%m-%d %H:%M:%S"

    # Leemos únicamente las columnas necesarias
    datos = pd.read_csv(
        archivo,
        encoding="latin1",
        usecols=["Inicio_del_viaje", "Fin_del_viaje"]
    )

    # Convertimos las fechas utilizando el formato correspondiente
    inicio = pd.to_datetime(
        datos["Inicio_del_viaje"],
        format=formato_fecha,
        errors="coerce"
    )

    fin = pd.to_datetime(
        datos["Fin_del_viaje"],
        format=formato_fecha,
        errors="coerce"
    )

    # Registramos las incidencias encontradas
    resultados_temporales.append({
        "Archivo": archivo.name,
        "Inicio inválido": inicio.isna().sum(),
        "Fin inválido": fin.isna().sum(),
        "Duración negativa": (
            (fin < inicio) & inicio.notna() & fin.notna()
        ).sum()
    })

# Consolidamos los resultados
resumen_temporal_final = pd.DataFrame(resultados_temporales)

# Presentamos los indicadores de calidad
print("Archivos revisados:", len(resumen_temporal_final))

print("\nTotal de incidencias:")
print(
    resumen_temporal_final[
        ["Inicio inválido", "Fin inválido", "Duración negativa"]
    ].sum()
)

# Mostramos los archivos que presenten alguna incidencia
resumen_temporal_final[
    resumen_temporal_final[
        ["Inicio inválido", "Fin inválido", "Duración negativa"]
    ].sum(axis=1) > 0
]

Archivos revisados: 24

Total de incidencias:
Inicio inválido      0
Fin inválido         0
Duración negativa    0
dtype: int64


,Archivo,Inicio inválido,Fin inválido,Duración negativa


In [21]:
# Reunimos los identificadores de viaje de los 24 archivos
identificadores = []

for archivo in archivos:

    # Leemos únicamente la columna de identificadores
    datos = pd.read_csv(
        archivo,
        encoding="latin1",
        usecols=["Viaje_Id"]
    )

    identificadores.append(datos["Viaje_Id"])

# Unimos los identificadores de todos los meses
todos_los_ids = pd.concat(
    identificadores,
    ignore_index=True
)

# Comprobamos si existen identificadores repetidos entre archivos
duplicados_globales = todos_los_ids.duplicated().sum()

print("Identificadores examinados:", len(todos_los_ids))
print("Identificadores duplicados:", duplicados_globales)

Identificadores examinados: 9177953
Identificadores duplicados: 0


In [22]:
# Creamos una carpeta para conservar los resultados de la auditoría
from pathlib import Path

carpeta_reportes = Path("reportes_validacion")
carpeta_reportes.mkdir(exist_ok=True)

# Exportamos los resultados mensuales de calidad de datos
resumen_mensual.to_csv(
    carpeta_reportes / "resumen_calidad_mensual.csv",
    index=False,
    encoding="utf-8-sig"
)

# Exportamos los resultados de la validación temporal
resumen_temporal_final.to_csv(
    carpeta_reportes / "resumen_validacion_temporal.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardamos el resumen anual
resumen_anual.to_csv(
    carpeta_reportes / "resumen_calidad_anual.csv",
    encoding="utf-8-sig"
)

print("Reportes generados en:", carpeta_reportes.resolve())

Reportes generados en: C:\Users\L03108121\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Documents\MNA\MLOPS\proyecto\datos_2024_2025\reportes_validacion
